In [0]:
# ============================================
# 补充 3: Broadcast Join 性能优化
# ============================================

from pyspark.sql import functions as F

cur_base = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/"

fact_sales = spark.read.format("delta").load(cur_base + "FactSales/")
dim_territory = spark.read.format("delta").load(cur_base + "DimTerritory/")

print(f"FactSales rows: {fact_sales.count()}")
print(f"DimTerritory rows: {dim_territory.count()}")

FactSales rows: 121317
DimTerritory rows: 10


In [0]:
# 普通 JOIN
normal_join = (fact_sales.alias("f")
    .join(dim_territory.alias("t"), 
          F.col("f.TerritoryKey") == F.col("t.TerritoryKey"), 
          how="left")
    .groupBy("TerritoryName")
    .agg(F.sum("LineTotal").alias("TotalSales"))
)

normal_join.explain(True)   # ⭐ 看执行计划
normal_join.show()

== Parsed Logical Plan ==
'Aggregate ['TerritoryName], ['TerritoryName, 'sum('LineTotal) AS TotalSales#22074]
+- 'Join LeftOuter, '`==`('f.TerritoryKey, 't.TerritoryKey)
   :- SubqueryAlias f
   :  +- Relation [DateKey#22006,CustomerKey#22007L,ProductKey#22008L,SalesPersonKey#22009L,TerritoryKey#22010L,SalesOrderID#22011,SalesOrderDetailID#22012,OrderQty#22013,UnitPrice#22014,UnitPriceDiscount#22015,LineTotal#22016] parquet
   +- SubqueryAlias t
      +- Relation [TerritoryKey#22042L,TerritoryID#22043,TerritoryName#22044,CountryRegionCode#22045,GroupName#22046] parquet

== Analyzed Logical Plan ==
TerritoryName: string, TotalSales: decimal(29,4)
Aggregate [TerritoryName#22044], [TerritoryName#22044, sum(LineTotal#22016) AS TotalSales#22074]
+- Join LeftOuter, (TerritoryKey#22010L = TerritoryKey#22042L)
   :- SubqueryAlias f
   :  +- Relation [DateKey#22006,CustomerKey#22007L,ProductKey#22008L,SalesPersonKey#22009L,TerritoryKey#22010L,SalesOrderID#22011,SalesOrderDetailID#22012,OrderQty

In [0]:
# Broadcast JOIN
broadcast_join = (fact_sales.alias("f")
    .join(F.broadcast(dim_territory).alias("t"), 
          F.col("f.TerritoryKey") == F.col("t.TerritoryKey"), 
          how="left")
    .groupBy("TerritoryName")
    .agg(F.sum("LineTotal").alias("TotalSales"))
)

broadcast_join.explain(True)   # ⭐ 看执行计划
broadcast_join.show()

== Parsed Logical Plan ==
'Aggregate ['TerritoryName], ['TerritoryName, 'sum('LineTotal) AS TotalSales#22127]
+- 'Join LeftOuter, '`==`('f.TerritoryKey, 't.TerritoryKey)
   :- SubqueryAlias f
   :  +- Relation [DateKey#22006,CustomerKey#22007L,ProductKey#22008L,SalesPersonKey#22009L,TerritoryKey#22010L,SalesOrderID#22011,SalesOrderDetailID#22012,OrderQty#22013,UnitPrice#22014,UnitPriceDiscount#22015,LineTotal#22016] parquet
   +- SubqueryAlias t
      +- UnresolvedHint broadcast
         +- Relation [TerritoryKey#22042L,TerritoryID#22043,TerritoryName#22044,CountryRegionCode#22045,GroupName#22046] parquet

== Analyzed Logical Plan ==
TerritoryName: string, TotalSales: decimal(29,4)
Aggregate [TerritoryName#22044], [TerritoryName#22044, sum(LineTotal#22016) AS TotalSales#22127]
+- Join LeftOuter, (TerritoryKey#22010L = TerritoryKey#22042L)
   :- SubqueryAlias f
   :  +- Relation [DateKey#22006,CustomerKey#22007L,ProductKey#22008L,SalesPersonKey#22009L,TerritoryKey#22010L,SalesOrderID#22

In [0]:
# 确认两次结果一致
print("=== Normal Join ===")
normal_join.show()

print("=== Broadcast Join ===")
broadcast_join.show()

=== Normal Join ===
+--------------+-------------+
| TerritoryName|   TotalSales|
+--------------+-------------+
|     Australia|10655335.9611|
|United Kingdom| 7670721.0375|
|     Southwest|24184609.6047|
|        Canada|16355770.4586|
|     Northwest|16084942.5513|
|       Central| 7909009.0073|
|     Northeast| 6939374.4830|
|        France| 7251555.6493|
|       Germany| 4915407.5975|
|     Southeast| 7879655.0747|
+--------------+-------------+

=== Broadcast Join ===
+--------------+-------------+
| TerritoryName|   TotalSales|
+--------------+-------------+
|     Australia|10655335.9611|
|United Kingdom| 7670721.0375|
|     Southwest|24184609.6047|
|        Canada|16355770.4586|
|     Northwest|16084942.5513|
|       Central| 7909009.0073|
|     Northeast| 6939374.4830|
|        France| 7251555.6493|
|       Germany| 4915407.5975|
|     Southeast| 7879655.0747|
+--------------+-------------+

